# TD - Financial Reporting

Lisez bien le [README.md](README.md) avant de commencer 📖

> **VS Code** : ouvrez le fichier puis `CTRL+SHIFT+V` pour le prévisualiser 

In [ ]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.2 s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

df = pd.read_parquet(
    "https://minio.lab.sspcloud.fr/fabienhos/td-reporting-financial/financial_data.parquet"
)

df.info()
df.head()

### Inspectez la présence de valeurs manquantes

In [ ]:
df.isna().sum()

#### Questions
1. Quelle stratégie adopter ?
2. Quelles sont les hypothèses que vous pourriez émettre afin d'expliquer ces valeurs manquantes ?
3. Comment pourriez-vous les vérifier ?

---

1. Stratégie : On ne supprime PAS les données sans réfléchir / On peut passer à côté d'une information. Une information vide n'est pas forcément mauvaise.
**L'absence d'information est une information en soit.**

2. Hypothèses :
- H1. Il est possible que les clients ne possèdent pas encore de score ou qu'un problème technique empêche le scoring
- H2. pour score_prev, il est possible que les clients soient nouveaux
- H3. Il est d'ailleurs possible que ceux avec un score manquant ne possèdent pas de score_prev non plus


3. En explorant 📊🔍

### Explorez les variables disponibles

Utilisez des counts, des tableaux croisés, des graphiques ...

```python
df['type_client'].value_counts()
pd.crosstab(df['score'], df['score_prev'])
```

In [ ]:
import matplotlib.pyplot as plt
fig , ax = plt.subplots(2, 2, figsize=(15, 10))

# Répartition PP / PM
df['type_client'].value_counts().plot(
    kind='bar', ax=ax[0, 0], title='Répartition des types de clients'
)
ax[0, 0].set_xticklabels(ax[0, 0].get_xticklabels(), rotation=0)

# Score actuel vs score précédent
color_mapping = {'V': '#2ecc71', 'O': '#f39c12', 'R': '#e74c3c'}
cross_tab = pd.crosstab(df['score'], df['score_prev'])
cross_tab.plot(
    kind='bar',
    stacked=True,
    ax=ax[0, 1],
    color=[color_mapping[col] for col in cross_tab.columns],
    title='Relation score actuel vs précédent',
    edgecolor='white',
    linewidth=0.5,
)
ax[0, 1].legend(title='Score précédent')

# Répartition des agents
df['id_agent'].value_counts().plot(
    kind='pie', ax=ax[1, 0], autopct='%1.1f%%', title='Répartition des agents'
)

# Complétude DRC
df['drc_complet'].value_counts().plot(
    kind='bar', ax=ax[1, 1], title='Complétude DRC'
)
ax[1, 1].set_xticklabels(['Complet', 'Incomplet'], rotation=0)

plt.tight_layout()
plt.show()

print(f"Première adhésion : {df['date_adhesion'].min()}")
print(f"Dernière adhésion  : {df['date_adhesion'].max()}")
print("\nRépartition des scores :")
print(pd.crosstab(df['score'], df['score_prev'], margins=True))

Rédigez vos observations sur les variables ici

Observations :

### Traitez les données nécessaires

In [ ]:
df = (
    df
    .assign(
        score=df['score'].fillna('S'),
        score_prev=df['score_prev'].fillna('N'),
        id_agent=df['id_agent'].where(df['id_agent'] == 'AUTO', 'MANUEL'),
    )
)

df

### Génération du rapport Excel

In [33]:
path_file = "template/template_reporting.xlsx"

with pd.ExcelWriter(
    path_file, mode="a", engine="openpyxl", if_sheet_exists="replace"
) as writer:
    df.to_excel(writer, sheet_name="DATA", index=False)

Si vous téléchargez le fichier Excel depuis `template/template_reporting.xlsx` vous constaterez qu'une page contenant le DataFrame a été ajouté. 

Nous sommes dans une situation idéale pour construire nos indicateurs 👀

In [ ]:
from openpyxl import load_workbook

wb = load_workbook(path_file)
ws = wb['Indicateurs']

# Répartition PP/PM
ws['E8'] = '=COUNTIF(DATA!B:B, "PP")'
ws['E9'] = '=COUNTIF(DATA!B:B, "PM")'
ws['E10'] = '=SUM(E8:E9)'

# Scores V/O/R pour les PP
ws['E14'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "V")'
ws['E15'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "O")'
ws['E16'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "R")'
ws['E17'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "M")'

# DRC Complet
ws['E22'] = '=COUNTIFS(DATA!B:B, "PP", DATA!G:G, "VRAI")'
ws['E23'] = '=COUNTIFS(DATA!B:B, "PM", DATA!G:G, "VRAI")'
ws['E24'] = '=SUM(E22:E23)'

# Focus V/O
ws['E28'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "V")'
ws['E29'] = '=COUNTIFS(DATA!B:B, "PM", DATA!D:D, "V")'
ws['E30'] = '=SUM(E28:E29)'

ws['E31'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "O")'
ws['E32'] = '=COUNTIFS(DATA!B:B, "PM", DATA!D:D, "O")'
ws['E33'] = '=SUM(E31:E32)'

# Focus V/O avec DRC complet
ws['E34'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "V", DATA!G:G, "VRAI")'
ws['E35'] = '=COUNTIFS(DATA!B:B, "PM", DATA!D:D, "V", DATA!G:G, "VRAI")'
ws['E36'] = '=SUM(E34:E35)'
ws['E37'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "O", DATA!G:G, "VRAI")'
ws['E38'] = '=COUNTIFS(DATA!B:B, "PM", DATA!D:D, "O", DATA!G:G, "VRAI")'
ws['E39'] = '=SUM(E37:E38)'

# Focus R
ws['E43'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "R")'
ws['E44'] = '=COUNTIFS(DATA!B:B, "PM", DATA!D:D, "R")'
ws['E45'] = '=SUM(E43:E44)'

ws['E46'] = '=COUNTIFS(DATA!D:D, "R", DATA!F:F, "AUTO")'
ws['E47'] = '=COUNTIFS(DATA!D:D, "R", DATA!F:F, "MANUEL")'
ws['E48'] = '=SUM(E46:E47)'

# R avec DRC complet
ws['E49'] = '=COUNTIFS(DATA!B:B, "PP", DATA!D:D, "R", DATA!G:G, "VRAI")'
ws['E50'] = '=COUNTIFS(DATA!B:B, "PM", DATA!D:D, "R", DATA!G:G, "VRAI")'
ws['E51'] = '=SUM(E49:E50)'

# Nouveaux clients (score_prev = Z) avec DRC complet
ws['E52'] = '=COUNTIFS(DATA!B:B, "PP", DATA!E:E, "Z", DATA!G:G, "VRAI")'
ws['E53'] = '=COUNTIFS(DATA!B:B, "PM", DATA!E:E, "Z", DATA!G:G, "VRAI")'
ws['E54'] = '=SUM(E52:E53)'

wb.save(path_file)
wb.close()

### Commentaire

Comme vous pouvez le voir, remplir ce fichier ne sera pas un soucis, c'est très simple de copier/coller la formule X fois en l'adaptant à la question.

Je vous invite à prendre un peu de recul sur ce que vous faites et à chercher s'il n'existerait pas une solution plus adapté.

Indices : data structures, fonctions, dataclass ... ?

In [ ]:
def fill_indicators(path_file: str, data_sheet: str ='DATA') -> None:
    """
    Fill the indicators in the Excel file with formulas.

    Args:
        path_file (str): Path to the Excel file.
        data_sheet (str, optional): Name of the data sheet. Defaults to 'DATA'.
    """
    wb = load_workbook(path_file)
    ws = wb['Indicateurs']
    
    config = [
        # Répartition PP/PM
        {'row': 8, 'formule': 'COUNTIF', 'args': [('B', 'PP')]},
        {'row': 9, 'formule': 'COUNTIF', 'args': [('B', 'PM')]},
        {'row': 10, 'formule': 'SUM', 'args': ['E8:E9']},
        
        # Scores V/O/R
        {'row': 14, 'formule': 'COUNTIFS', 'args': [('B', 'PP'), ('D', 'V')]},
        {'row': 15, 'formule': 'COUNTIFS', 'args': [('B', 'PP'), ('D', 'O')]},
        {'row': 16, 'formule': 'COUNTIFS', 'args': [('B', 'PP'), ('D', 'R')]},
        {'row': 17, 'formule': 'COUNTIFS', 'args': [('B', 'PP'), ('D', 'M')]},
        
        # DRC Complet
        {'row': 22, 'formule': 'COUNTIFS', 'args': [('B', 'PP'), ('G', "VRAI")]},
        {'row': 23, 'formule': 'COUNTIFS', 'args': [('B', 'PM'), ('G', "VRAI")]},
        {'row': 24, 'formule': 'SUM', 'args': ['E22:E23']},
        # Focus V/O avec DRC
        {'row': 28, 'formule': 'COUNTIFS', 'args': [('B', 'PP'), ('D', 'V')]},
        {'row': 29, 'formule': 'COUNTIFS', 'args': [('B', 'PM'), ('D', 'V')]},
        {'row': 30, 'formule': 'SUM', 'args': ['E28:E29']},
    ]

    for item in config:
        cell = f'E{item["row"]}'
        
        if item['formule'] == 'COUNTIF':
            col, val = item['args'][0]
            ws[cell] = f'=COUNTIF({data_sheet}!{col}:{col}, "{val}")'
            
        elif item['formule'] == 'COUNTIFS':
            criteria = [f'{data_sheet}!{col}:{col}, "{val}"' for col, val in item['args']]
            ws[cell] = f'=COUNTIFS({", ".join(criteria)})'
            
        elif item['formule'] == 'SUM':
            ws[cell] = f'=SUM({item["args"][0]})'

    wb.save(path_file)
    wb.close()

# Lancer la fonction pour remplir les indicateurs
fill_indicators(path_file)